In [ ]:
# ============================================================
# RANDOM FOREST — BIOMASS PREDICTION
# UAV Multispectral | Winter Wheat | Virginia (3 sites)
# Run each cell separately in Jupyter Notebook
# ============================================================


# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.feature_selection import mutual_info_regression
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection    import KFold, RandomizedSearchCV
from sklearn.ensemble           import RandomForestRegressor
from sklearn.preprocessing      import StandardScaler
from sklearn.metrics            import r2_score, mean_squared_error
from sklearn.inspection         import permutation_importance
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.patches         import Patch
import warnings

warnings.filterwarnings("ignore")
print("All libraries imported successfully.")

In [ ]:
# ============================================================
# CELL 2 — FILE PATHS AND COLUMN NAMES
# Update these to match your actual file paths and column names
# ============================================================

# Input CSV
file_path = r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Python1\biomass_with_thermal_time_F1.csv'

# Output folder — will be created automatically
output_dir = Path(r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Python1\Result_Chapter_3\RF\Biomass_tha')
output_dir.mkdir(parents=True, exist_ok=True)


# Column names — update if your CSV uses different names
category_col = 'Growth_Stage'       # column with stage names
location_col = 'Location'           # column with site names
target_var   = 'Biomass_density(t_ha)'  # target variable
dap_col      = 'DAP'                # days after planting
gdd_col      = 'GDD'                # growing degree days

print(f"Output directory: {output_dir}")

# ============================================================
# CELL 3 — LOAD AND INSPECT DATA
# ============================================================
 
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
 
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nGrowth stages found: {df[category_col].unique()}")
print(f"Locations found:     {df[location_col].unique()}")
print(f"Target variable stats:\n{df[target_var].describe()}")
print(f"\nMissing values per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# One-hot encode Location column
df = pd.get_dummies(df, columns=['Location'], prefix='loc')

# Update complementary to use the actual dummy column names
# (run this after Cell 6 defines complementary)

# ============================================================
# CELL 4 — STAGE ORDER AND AESTHETICS
# ============================================================
 
stage_order = {
    'Tillering':           1,
    'Booting':             2,
    'Heading':             3,
    'Ripening_Harvesting': 4
}
 
valid_stages = sorted(
    df[category_col].unique(),
    key=lambda x: stage_order.get(x, 99)
)
print(f"Stages in order: {valid_stages}")
 
# Plot aesthetics
plt.rcParams['font.family']     = 'serif'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color']     = 'black'
plt.rcParams['ytick.color']     = 'black'
plt.rcParams['axes.edgecolor']  = 'black'
sns.set_style("ticks")
print("Aesthetics set.")

In [ ]:
# ============================================================
# CELL 6 — FEATURE POOLS
# ============================================================

vi_pool = [
    'Green','NIR','RE','Red','GNDVI','GRVI','MCARI','MSAVI','NDRE','NDVI',
    'NGRDI','OSAVI','REDVI','RVI','SAVI','TNDVI','TVI','WDRVI'
]

glcm_pool = [
    'Green_GLCM_contrast','Green_GLCM_correlation','Green_GLCM_dissimilarity',
    'Green_GLCM_entropy','Green_GLCM_homogeneity','Green_GLCM_mean',
    'Green_GLCM_second_moment','Green_GLCM_variance',
    'NIR_GLCM_contrast','NIR_GLCM_correlation','NIR_GLCM_dissimilarity',
    'NIR_GLCM_entropy','NIR_GLCM_homogeneity','NIR_GLCM_mean',
    'NIR_GLCM_second_moment','NIR_GLCM_variance',
    'RE_GLCM_contrast','RE_GLCM_correlation','RE_GLCM_dissimilarity',
    'RE_GLCM_entropy','RE_GLCM_homogeneity','RE_GLCM_mean',
    'RE_GLCM_second_moment','RE_GLCM_variance',
    'Red_GLCM_contrast','Red_GLCM_correlation','Red_GLCM_dissimilarity',
    'Red_GLCM_entropy','Red_GLCM_homogeneity','Red_GLCM_mean',
    'Red_GLCM_second_moment','Red_GLCM_variance',
]

ndti_pool = [
    'NDTI_Green-Red_contrast','NDTI_Green-Red_correlation','NDTI_Green-Red_dissimilarity',
    'NDTI_Green-Red_entropy','NDTI_Green-Red_homogeneity','NDTI_Green-Red_mean',
    'NDTI_Green-Red_second_moment','NDTI_Green-Red_variance',
    'NDTI_NIR-Green_contrast','NDTI_NIR-Green_correlation','NDTI_NIR-Green_dissimilarity',
    'NDTI_NIR-Green_entropy','NDTI_NIR-Green_homogeneity','NDTI_NIR-Green_mean',
    'NDTI_NIR-Green_second_moment','NDTI_NIR-Green_variance',
    'NDTI_NIR-RE_contrast','NDTI_NIR-RE_correlation','NDTI_NIR-RE_dissimilarity',
    'NDTI_NIR-RE_entropy','NDTI_NIR-RE_homogeneity','NDTI_NIR-RE_mean',
    'NDTI_NIR-RE_second_moment','NDTI_NIR-RE_variance',
    'NDTI_NIR-Red_contrast','NDTI_NIR-Red_correlation','NDTI_NIR-Red_dissimilarity',
    'NDTI_NIR-Red_entropy','NDTI_NIR-Red_homogeneity','NDTI_NIR-Red_mean',
    'NDTI_NIR-Red_variance',
    'NDTI_RE-Green_contrast','NDTI_RE-Green_correlation','NDTI_RE-Green_dissimilarity',
    'NDTI_RE-Green_entropy','NDTI_RE-Green_homogeneity','NDTI_RE-Green_mean',
    'NDTI_RE-Green_second_moment','NDTI_RE-Green_variance',
    'NDTI_RE-Red_contrast','NDTI_RE-Red_correlation','NDTI_RE-Red_dissimilarity',
    'NDTI_RE-Red_entropy','NDTI_RE-Red_homogeneity','NDTI_RE-Red_mean',
    'NDTI_RE-Red_second_moment','NDTI_RE-Red_variance',
]

# Ancillary pool: DAP + GDD
ancillary = ['DAP', 'GDD']

print(f"VI pool size:   {len(vi_pool)}")
print(f"GLCM pool size: {len(glcm_pool)}")
print(f"NDTI pool size: {len(ndti_pool)}")
print(f"Ancillary pool (no location): {ancillary}")

In [ ]:
 
def make_scenario(pools):
    """Combine pools and deduplicate while preserving order."""
    combined = []
    seen     = set()
    for pool in pools:
        for feat in pool:
            if feat not in seen:
                seen.add(feat)
                combined.append(feat)
    return combined
 
scenarios = {
    '1. ancillary':          make_scenario([ancillary]),
    '2. VI + Anc':           make_scenario([vi_pool,   ancillary]),
    '3. GLCM + Anc':         make_scenario([glcm_pool, ancillary]),
    '4. NDTI + Anc':         make_scenario([ndti_pool, ancillary]),
    '5. VI + GLCM + Anc':    make_scenario([vi_pool,   glcm_pool, ancillary]),
    '6. VI + NDTI + Anc':    make_scenario([vi_pool,   ndti_pool, ancillary]),
    '7. GLCM + NDTI + Anc':  make_scenario([glcm_pool, ndti_pool, ancillary]),
    '8. VI+GLCM+NDTI + Anc': make_scenario([vi_pool,   glcm_pool, ndti_pool, ancillary]),
}
 
print("Scenarios built:")
for name, pool in scenarios.items():
    print(f"  {name:<28} → {len(pool)} features")
    # ============================================================
# CELL 8 — FIXED HYPERPARAMETERS AND KFOLD
# ============================================================

rf_params = {
    'n_estimators':      500,
    'max_depth':         10,
    'min_samples_split': 10,
    'min_samples_leaf':  10,
    'max_features':      'sqrt',
    'random_state':      42,
    'n_jobs':            -1
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Fixed RF hyperparameters:")
for k, v in rf_params.items():
    print(f"  {k}: {v}")
print(f"\nKFold: n_splits=5, shuffle=True, random_state=42")

In [ ]:
# ==========================================
# MI-VIF FUNCTIONS
# Both run on TRAINING DATA ONLY — no leakage
# ==========================================

def step_1_mi_filter(X_train_df, y_train, top_n=30, random_state=42):
    X_train_df = X_train_df.astype(float).fillna(0.0)
    
    # Calculate scores
    mi_scores = mutual_info_regression(
        X_train_df, y_train, random_state=random_state
    )
    
    # Create sorted series
    mi_series = pd.Series(mi_scores, index=X_train_df.columns).sort_values(ascending=False)
    
    # Identify top N features
    n_keep = min(top_n, len(mi_series))
    top_mi_series = mi_series.head(n_keep)
    
    # Create the list of names for Step 2
    selected_names = list(top_mi_series.index)
    
    # IMPORTANT: Return the LIST first, then the SERIES
    return selected_names, top_mi_series

def step_2_vif_filter(X_train_df, selected_features, vif_threshold=10.0):
    features = selected_features.copy()
    while len(features) > 1:
        X_sub = X_train_df[features].copy()

        # ── ADD THESE TWO LINES ──────────────────────────────
        # Force all columns to float and drop any remaining NaN
        X_sub = X_sub.astype(float).fillna(0.0)
        # ────────────────────────────────────────────────────

        for col in X_sub.columns:
            if X_sub[col].std() == 0:
                X_sub[col] += np.random.normal(0, 1e-8, len(X_sub))
        try:
            vif_values = [
                variance_inflation_factor(X_sub.values, i)
                for i in range(X_sub.shape[1])
            ]
        except Exception:
            break
        vif_df = pd.DataFrame({
            'Feature': features,
            'VIF':     vif_values
        }).sort_values('VIF', ascending=False)
        if vif_df.iloc[0]['VIF'] <= vif_threshold:
            break
        features.remove(vif_df.iloc[0]['Feature'])
    return features

def get_top_n(pool_size):
    if pool_size <= 20:  return min(15, pool_size)
    if pool_size <= 55:  return 25
    return 30

print("MI-VIF functions defined and ready.")


In [ ]:
# ==========================================
# MAIN RF LOOP WITH MI-VIF: UNIFIED VERSION
# ==========================================

results_log = []
stage_oof   = {}

# Header matches your requested format
print(f"\n{'Stage':<22} | {'Scenario':<28} | {'N':<5} | "
      f"{'MI_kept':<8} | {'MI_Min':<7} | {'VIF_kept':<9} | "
      f"{'R2_C':<5} | {'R2_V':<5} | {'RMSE_C':<7} | {'RMSE_V':<7} | {'rRMSE%':<7} | {'RPD':<5}")
print("-" * 155)

for stage in valid_stages:
    stage_df = df[df[category_col] == stage].copy().reset_index(drop=True)

    for sc_name, pool in scenarios.items():
        valid_pool = [
            p for p in pool 
            if p in stage_df.columns and stage_df[p].nunique() > 1
        ]
        sub_df = stage_df.dropna(
            subset=valid_pool + [target_var]
        ).reset_index(drop=True)

        if len(sub_df) < 15:
            continue

        X_full = sub_df[valid_pool]
        y_full = sub_df[target_var]
        top_n = get_top_n(len(valid_pool))

        # Accumulators for metrics and plotting
        fold_r2c, fold_r2v     = [], []
        fold_rmsec, fold_rmsev   = [], []
        fold_rpd               = []
        fold_mi_kept, fold_vif_kept = [], []
        fold_final_features    = []
        fold_mi_min_scores     = [] 
        fold_importances       = []

        # OOF Prediction storage for plotting
        oof_obs  = np.full(len(sub_df), np.nan)
        oof_pred = np.full(len(sub_df), np.nan)
        cal_obs_all, cal_pred_all = [], []

        for train_idx, val_idx in kf.split(X_full):
            X_tr_raw = X_full.iloc[train_idx].reset_index(drop=True)
            y_tr     = y_full.iloc[train_idx].reset_index(drop=True)
            X_val_raw= X_full.iloc[val_idx].reset_index(drop=True)
            y_val    = y_full.iloc[val_idx].reset_index(drop=True)

            # ── STEP 1: MI FILTER ──────────────────────
            mi_features, mi_series = step_1_mi_filter(
                X_tr_raw, y_tr.values,
                top_n=top_n, random_state=42
            )
            
            if not mi_series.empty:
                fold_mi_min_scores.append(mi_series.iloc[-1])

            # ── STEP 2: VIF FILTER ─────────────────────
            final_features = step_2_vif_filter(
                X_tr_raw, mi_features, vif_threshold=10.0
            )

            if not final_features:
                final_features = [mi_series.index[0]]

            fold_mi_kept.append(len(mi_features))
            fold_vif_kept.append(len(final_features))
            fold_final_features.append(final_features)

            # ── STEP 3: MODELING ───────────────────────
            scaler   = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X_tr_raw[final_features])
            X_val_sc = scaler.transform(X_val_raw[final_features])

            rf = RandomForestRegressor(**rf_params)
            rf.fit(X_tr_sc, y_tr.values)

            y_cal_pred = rf.predict(X_tr_sc)
            y_val_pred = rf.predict(X_val_sc)

            # Collect metrics
            fold_r2c.append(r2_score(y_tr, y_cal_pred))
            fold_r2v.append(max(r2_score(y_val, y_val_pred), 0.0))
            fold_rmsec.append(np.sqrt(mean_squared_error(y_tr, y_cal_pred)))
            fold_rmsev.append(np.sqrt(mean_squared_error(y_val, y_val_pred)))
            fold_rpd.append(np.std(y_val.values) / np.sqrt(mean_squared_error(y_val, y_val_pred)))

            # Collect Feature Importances (Padded for valid_pool consistency)
            imp_full = np.zeros(len(valid_pool))
            for j, feat in enumerate(final_features):
                if feat in valid_pool:
                    imp_full[valid_pool.index(feat)] = rf.feature_importances_[j]
            fold_importances.append(imp_full)

            # Collect OOF predictions
            oof_obs[val_idx]  = y_val.values
            oof_pred[val_idx] = y_val_pred
            cal_obs_all.extend(y_tr.values)
            cal_pred_all.extend(y_cal_pred)

        # Average metrics
        r2c_m   = np.mean(fold_r2c)
        r2v_m   = np.mean(fold_r2v)
        rmsec_m = np.mean(fold_rmsec)
        rmsev_m = np.mean(fold_rmsev)
        rpd_m   = np.mean(fold_rpd)
        mi_min_m = np.mean(fold_mi_min_scores) if fold_mi_min_scores else 0
        rrmse   = (rmsev_m / np.mean(y_full) * 100) if np.mean(y_full) > 0 else np.nan

        # Print progress
        print(
            f"{stage:<22} | {sc_name:<28} | {len(sub_df):<5} | "
            f"{np.mean(fold_mi_kept):<8.1f} | {mi_min_m:<7.3f} | {np.mean(fold_vif_kept):<9.1f} | "
            f"{r2c_m:<5.2f} | {r2v_m:<5.2f} | {rmsec_m:<7.3f} | {rmsev_m:<7.3f} | {rrmse:<7.1f} | {rpd_m:<5.2f}"
        )

        # Identify Stable Features
        all_feats = [f for fold in fold_final_features for f in fold]
        feat_counts = pd.Series(all_feats).value_counts()
        stable_feats = list(feat_counts[feat_counts >= 3].index)
        
        # Log results (Matches your Cell 10 naming exactly)
        results_log.append({
            'Stage': stage,
            'Scenario': sc_name,
            'N': len(sub_df),
            'Avg_MI_kept': round(mi_min_m, 4), # Preserving your name here
            'Avg_VIF_kept': round(np.mean(fold_vif_kept), 1),
            'R2_Cal': round(r2c_m, 4),
            'R2_Val': round(r2v_m, 4),
            'RMSE_Cal': round(rmsec_m, 4),
            'RMSE_Val': round(rmsev_m, 4),
            'rRMSE_Val_%': round(rrmse, 2), # Preserving your name here
            'RPD': round(rpd_m, 4),
            'Stable_Features': ", ".join(stable_feats),
            'Cal_Val_Gap': round(r2c_m - r2v_m, 3)
        })

        # Populate stage_oof for Cell 11 and Cell 12
        valid_mask = ~np.isnan(oof_pred)
        stage_oof[f"{stage}||{sc_name}"] = {
            'val_obs':   oof_obs[valid_mask],
            'val_pred':  oof_pred[valid_mask],
            'cal_obs':   np.array(cal_obs_all),
            'cal_pred':  np.array(cal_pred_all),
            'r2v': r2v_m, 'r2c': r2c_m,
            'rmsev': rmsev_m, 'rmsec': rmsec_m,
            'imp_names': valid_pool,
            'imp_vals': np.mean(fold_importances, axis=0),
            'scenario': sc_name,
            'stage': stage
        }

print("\nMain loop complete. Metrics logged and stage_oof populated.")

In [ ]:
# ============================================================
# CELL 10 — SAVE RESULTS TABLE TO CSV
# ============================================================

df_results = pd.DataFrame(results_log)

csv_path = output_dir / 'Biomass_RF_MIVIF_Results.csv'
df_results.to_csv(csv_path, index=False)

print(f"Results saved to: {csv_path}")
print(f"\nShape: {df_results.shape}")
print(f"\nPreview:")
display(df_results[[
    'Stage', 'Scenario', 'N',
    'Avg_MI_kept', 'Avg_VIF_kept',
    'R2_Cal', 'R2_Val',
    'RMSE_Cal', 'RMSE_Val',
    'rRMSE_Val_%', 'RPD',
    'Cal_Val_Gap', 'Stable_Features'
]])

# Feature reduction summary
print("\nFeature reduction summary (avg across folds):")
print(df_results.groupby('Stage')[['Avg_MI_kept','Avg_VIF_kept']].mean().round(1))

In [ ]:
# ============================================================
# CELL 11 — IDENTIFY WINNER SCENARIO PER STAGE
# (excludes ancillary-only as it has no spectral features)
# ============================================================

winners = {}

for stage in valid_stages:
    stage_res = [
        r for r in results_log
        if r['Stage'] == stage
        and r['Scenario'] != '1. Ancillary only'   # ← exclude baseline
    ]
    if not stage_res:
        continue
    winner = max(stage_res, key=lambda x: x['R2_Val'])
    key    = f"{stage}||{winner['Scenario']}"
    if key in stage_oof:
        winners[stage] = stage_oof[key]

print("Winners (spectral scenarios only, highest R²_val):")
for stage, d in winners.items():
    print(f"  {stage:<25} → {d['scenario']:<35} "
          f"R²_val={d['r2v']:.3f}  R²_cal={d['r2c']:.3f}  "
          f"Gap={d['r2c']-d['r2v']:.3f}")


In [ ]:
# ============================================================
# CELL 12 — PLOT: OBSERVED VS PREDICTED (UNIFORM AXES)
# ============================================================

# 1. CALCULATE GLOBAL LIMITS ACROSS ALL STAGES
all_global_vals = []
for stage, d in winners.items():
    all_global_vals.extend([d['cal_obs'], d['cal_pred'], d['val_obs'], d['val_pred']])

# Flatten and find absolute min/max
all_global_vals = np.concatenate(all_global_vals)
g_min, g_max = all_global_vals.min(), all_global_vals.max()

# Apply a single 10% padding for the entire figure
pad = (g_max - g_min) * 0.1
global_lims = [g_min - pad, g_max + pad]

num_stages = len(winners)  
fig, axes  = plt.subplots(2, num_stages, figsize=(4*num_stages, 8), squeeze=False)

for i, (stage, d) in enumerate(winners.items()):
    
    # Clean up stage labels for the plot title
    lbl = stage.replace('_', '/').replace('Ripening/Harvesting', 'Ripening/\nHarvesting')

    for row, (obs, pred, color, label, r2, rmse) in enumerate([
        (d['cal_obs'], d['cal_pred'], 'darkorange', 'Calibration', d['r2c'], d['rmsec']),
        (d['val_obs'], d['val_pred'], 'royalblue',  'Validation',  d['r2v'], d['rmsev']),
    ]):
        ax = axes[row, i]
        
        # 1. Plot the 1:1 Reference Line using GLOBAL LIMS
        ax.plot(global_lims, global_lims, 'k--', lw=1.2, alpha=0.8, zorder=2)
        
        # 2. Scatter the points
        ax.scatter(obs, pred, facecolors=color, edgecolors='black',
                    s=35, alpha=0.7, linewidth=0.7, zorder=3)

        # Title and Stats
        title = f"{lbl}\n({label})" if row == 0 else f"({label})"
        ax.set_title(title, fontsize=12, fontweight='bold', color='black')

        r2_lbl = '$R^2_{cal}$' if row == 0 else '$R^2_{val}$'
        ax.text(0.05, 0.95,
                f"{r2_lbl}: {r2:.2f}\nRMSE: {rmse:.3f}",
                transform=ax.transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.75,
                          edgecolor='lightgrey', boxstyle='round,pad=0.3'))

        # Standard Axis Formatting
        ax.set_xlabel("Observed (t ha$^{-1}$)",  fontsize=12, color='black')
        ax.set_ylabel("Predicted (t ha$^{-1}$)", fontsize=12, color='black')
        
        # Force identical limits for all subplots
        ax.set_xlim(global_lims)
        ax.set_ylim(global_lims)
        
        ax.set_aspect('equal')
        ax.tick_params(colors='black', labelsize=12)
        for sp in ax.spines.values(): sp.set_edgecolor('black')

# Maintain Y-axis labeling on the leftmost plots
axes[0,0].set_ylabel("Predicted (t ha$^{-1}$)\n[Calibration]", fontsize=12, color='black')
axes[1,0].set_ylabel("Predicted (t ha$^{-1}$)\n[Validation]",  fontsize=12, color='black')

plt.tight_layout(pad=2.0)
plt.savefig(output_dir / 'Biomass_RF_MIVIF_ObsPred1.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Uniform axis plot saved using range: {global_lims}")

In [ ]:
# ============================================================
# CELL 13 — PLOT: RESIDUALS (UNIFORM Y, NATURAL X)
# ============================================================

# 1. CALCULATE GLOBAL RESIDUAL LIMITS (For Y-axis only)
all_residuals = []
for stage, d in winners.items():
    all_residuals.extend(d['cal_obs'] - d['cal_pred'])
    all_residuals.extend(d['val_obs'] - d['val_pred'])

# Centering the Y-axis at zero based on the absolute maximum error
abs_max_resid = np.abs(all_residuals).max() * 1.15 
global_resid_lims = [-abs_max_resid, abs_max_resid]

num_stages = len(winners)  
fig, axes = plt.subplots(2, num_stages, figsize=(4*num_stages, 8), squeeze=False)

for i, (stage, d) in enumerate(winners.items()):

    cal_resid = d['cal_obs'] - d['cal_pred']
    val_resid = d['val_obs'] - d['val_pred']
    lbl       = stage.replace('_', '/').replace('Ripening/Harvesting', 'Ripening/\nHarvesting')

    for row, (pred, resid, color, label) in enumerate([
        (d['cal_pred'], cal_resid, 'darkorange', 'Calibration'),
        (d['val_pred'], val_resid, 'royalblue',  'Validation'),
    ]):
        ax = axes[row, i]
        
        # 1. PLOT DATA POINTS
        ax.scatter(pred, resid, facecolors=color, edgecolors='black',
                   s=40, alpha=0.7, linewidth=0.7, zorder=3)
        
        # 2. ZERO REFERENCE LINE
        ax.axhline(0, color='black', lw=1.2, ls='--', zorder=2)

        # Calculate RMSE for the text box
        rmse_val = np.sqrt(mean_squared_error(
            d['cal_obs'] if row==0 else d['val_obs'],
            d['cal_pred'] if row==0 else d['val_pred']
        ))
        
        title = f"{lbl}\n({label} Residuals)" if row==0 else f"({label} Residuals)"
        ax.set_title(title, fontsize=12, fontweight='bold', color='black')
        
        # Annotate with bold RMSE
        ax.text(0.05, 0.95, f"RMSE: {rmse_val:.3f}",
                transform=ax.transAxes, 
                fontsize=11, 
                fontweight='bold', 
                verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.85,
                          edgecolor='lightgrey', boxstyle='round,pad=0.4'))
        
        ax.set_xlabel("Predicted (t ha$^{-1}$)", fontsize=12, color='black')
        ax.set_ylabel("Residual (t ha$^{-1}$)",  fontsize=12, color='black')
        
        # --- APPLY UNIFORM Y-LIMITS ---
        ax.set_ylim(global_resid_lims)
        
        # --- X-LIMIT IS NOW NATURAL (Auto-scaled by Matplotlib) ---
        # ax.set_xlim(global_lims) <-- Line removed to allow natural X-axis
        
        ax.tick_params(colors='black', labelsize=11)
        for sp in ax.spines.values(): sp.set_edgecolor('black')

# Labeling for the first column
axes[0,0].set_ylabel("Residual (t ha$^{-1}$)\n[Calibration]", fontsize=12, fontweight='bold', color='black')
axes[1,0].set_ylabel("Residual (t ha$^{-1}$)\n[Validation]",  fontsize=12, fontweight='bold', color='black')

plt.tight_layout(pad=2.5)
plt.savefig(output_dir / 'Biomass_RF_Residuals_UniformY_NaturalX.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Residual plot saved with uniform Y-range: {global_resid_lims} and natural X-axes.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ==========================================
# 1. SETUP & AESTHETICS
# ==========================================
plt.rcParams['font.family'] = 'serif'
sns.set_style("ticks")

# Added Ripening_Harvesting to the list
all_stages = ['Tillering', 'Booting', 'Heading', 'Ripening_Harvesting']

# Extended color and marker palettes
stage_colors = {
    'Tillering': '#33a02c', 
    'Booting': '#1f78b4', 
    'Heading': '#e31a1c',
    'Ripening_Harvesting': '#ff7f00' # Orange for harvest/senescence
}
stage_markers = {
    'Tillering': 'o', 
    'Booting': '^', 
    'Heading': 's',
    'Ripening_Harvesting': 'D' # Diamond for the final stage
}

# ==========================================
# 2. CONSOLIDATE DATA (All 4 Stages)
# ==========================================
cal_list, val_list = [], []

for stage in all_stages:
    if stage in winners:
        d = winners[stage]
        cal_list.append(pd.DataFrame({
            'Obs': d['cal_obs'], 
            'Pred': np.clip(d['cal_pred'], 0, None), 
            'Stage': stage
        }))
        val_list.append(pd.DataFrame({
            'Obs': d['val_obs'], 
            'Pred': np.clip(d['val_pred'], 0, None), 
            'Stage': stage
        }))

df_cal = pd.concat(cal_list).reset_index(drop=True)
df_val = pd.concat(val_list).reset_index(drop=True)

# ==========================================
# 3. PLOT FUNCTION (Fixed 0 to 3)
# ==========================================
def plot_rf_consolidated_4stage(ax, df_all, title, is_cal=True):
    lims = [0, 18.0]
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, zorder=2) # 1:1 Line
    
    handles = []
    for stage in all_stages:
        subset = df_all[df_all['Stage'] == stage]
        if subset.empty: continue
        
        # Plot points
        ax.scatter(subset['Obs'], subset['Pred'], 
                   facecolors=stage_colors[stage], edgecolors='black',
                   s=50, alpha=0.7, marker=stage_markers[stage], 
                   linewidth=0.7, zorder=3)
        
        # Pull metrics from your results dictionary
        metrics = winners[stage]
        r2 = metrics['r2c'] if is_cal else metrics['r2v']
        
        # Clean label for the legend
        clean_name = stage.replace('_', '/').title()
        lbl = f"{clean_name} ($R^2$: {r2:.2f})"
        
        h = plt.Line2D([0], [0], marker=stage_markers[stage], color='white',
                       markerfacecolor=stage_colors[stage], markeredgecolor='black',
                       markersize=9, linestyle='none', label=lbl)
        handles.append(h)

    # Axis Formatting
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel("Observed $Biomass$ (t ha$^{-1}$)", fontsize=12)
    ax.set_ylabel("Predicted $Biomass$ (t ha$^{-1}$)", fontsize=12)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xticks([0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20])
    ax.set_yticks([0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20])
    ax.set_aspect('equal')
    ax.grid(False)
    sns.despine(ax=ax)
    return handles

# ==========================================
# 4. GENERATE FIGURE
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))

h_cal = plot_rf_consolidated_4stage(axes[0], df_cal, "", is_cal=True)
h_val = plot_rf_consolidated_4stage(axes[1], df_val, "", is_cal=False)

# Add legends
axes[0].legend(handles=h_cal, loc='upper left', fontsize=12, frameon=True, title="Calibration Metrics")
axes[1].legend(handles=h_val, loc='upper left', fontsize=12, frameon=True, title="Validation Metrics")

plt.tight_layout(pad=3.0)
save_path = output_dir / 'Biomass_RF_Consolidated_ObsPred1.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Consolidated 4-Stage RF plot saved to: {save_path}")

In [ ]:
from matplotlib.patches import Patch
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CELL 14 — PLOT: FEATURE IMPORTANCE (Cleaned Titles)
# ============================================================

num_stages = len(winners)
fig, axes = plt.subplots(num_stages, 1, 
                         figsize=(8.5, 4.0 * num_stages), squeeze=False)

# Professional Palette
top_color = '#D85A30'       # Burnt Orange
secondary_color = '#378ADD' # Steel Blue

# Set font family to serif for publication quality
plt.rcParams['font.family'] = 'serif'

for i, (stage, d) in enumerate(winners.items()):
    ax = axes[i, 0]
    
    # 1. Clean up Stage Label
    lbl = stage.replace('_', ' ').title()
    if 'Ripening' in lbl: lbl = 'Ripening/Harvesting'
    
    # 2. Clean up Scenario Label (Removes "4. " from "4. NDTI_Anc")
    raw_scenario = d['scenario']
    if '.' in raw_scenario:
        # Split by period and take the part after the number
        clean_scenario = raw_scenario.split('.', 1)[1].strip()
    else:
        clean_scenario = raw_scenario
    
    # Replace underscores with plus for a cleaner "Data Fusion" look (e.g., NDTI+Anc)
    clean_scenario = clean_scenario.replace('_', '+')
    
    # 3. Extract data
    imp_vals  = d['imp_vals']
    imp_names = d['imp_names']

    # Sort and slice top 10
    top_idx   = np.argsort(imp_vals)[::-1][:10]
    top_names = [imp_names[j] for j in top_idx]
    top_vals  = [imp_vals[j]  for j in top_idx]

    # Assign colors: Top 5 warm, 6-10 cool
    colors = [top_color if j < 5 else secondary_color for j in range(len(top_vals))]

    # Plot horizontal bars
    bars = ax.barh(top_names[::-1], top_vals[::-1], 
                   color=colors[::-1], 
                   edgecolor='black', linewidth=0.8)

    # 4. Final Title Construction: "Tillering — NDTI+Anc (R2: 0.85)"
    ax.set_title(f"{lbl} — {clean_scenario} ($R^2_{{v}}$: {d['r2v']:.2f})", 
                 fontsize=13, fontweight='bold', color='black', pad=12)
    
    ax.set_xlabel("Mean Decrease in Impurity (Gini Importance)", 
                  fontsize=11, color='black')
    
    # Grid and Spine Formatting
    ax.tick_params(colors='black', labelsize=10)
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True) # Ensure grid is behind bars
    for sp in ax.spines.values(): 
        sp.set_edgecolor('black')

# Legend construction
legend_elements = [
    Patch(facecolor=top_color, edgecolor='black', label='Top 5 Primary Predictors'),
    Patch(facecolor=secondary_color, edgecolor='black', label='Supporting Predictors (6–10)'),
]

# Shared Legend at the bottom
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.01),
           edgecolor='lightgrey')

plt.tight_layout(pad=3.0)
# Dynamic adjustment based on the number of stages (3 or 4)
plt.subplots_adjust(bottom=0.08 if num_stages < 4 else 0.06)

# Save the figure
importance_path = output_dir / 'P_Uptake_RF_FeatureImportance_Clean.png'
plt.savefig(importance_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Cleaned feature importance plot saved to: {importance_path}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ============================================================
# CELL 15 — SCENARIO COMPARISON (Physically Removing 1st Bar)
# ============================================================

# Sorting stages chronologically
ordered_stages = ['Tillering', 'Booting', 'Heading', 'Ripening_Harvesting']
stages_found = [s for s in ordered_stages if s in set(r['Stage'] for r in results_log)]

fig, axes = plt.subplots(1, len(stages_found),
                         figsize=(4.5 * len(stages_found), 5.5),
                         squeeze=False)

# Mapping for colors based on the visual logic in your image
colors_dict = {
    'vi': '#D85A30',       # Burnt Orange
    'texture': '#378ADD',  # Steel Blue
    'ndti': '#888780',     # Grey
    'fusion': '#534AB7'    # Purple/Indigo
}

for i, stage in enumerate(stages_found):
    ax = axes[0, i]
    # Filter for this stage
    stage_res = [r for r in results_log if r['Stage'] == stage]
    
    # --- PHYSICAL REMOVAL OF 1ST BAR ---
    # We skip the very first scenario (Ancillary Only) 
    # to start the graph directly with sensor data.
    filtered_res = stage_res[1:] if len(stage_res) > 1 else stage_res
    
    if not filtered_res:
        continue

    r2v_vals = [r['R2_Val'] for r in filtered_res]
    raw_names = [r['Scenario'] for r in filtered_res]
    
    clean_labels = []
    bar_colors = []

    for name in raw_names:
        # 1. Clean Labels: Remove "1. " / "2. " and fix underscores
        temp_name = name.split('.', 1)[1].strip() if '.' in name else name
        temp_name = temp_name.replace('_', '+')
        
        # Ensure we keep the "Ancillary" text as requested
        temp_name = temp_name.replace('Anc', 'Anc') 
        clean_labels.append(temp_name)

        # 2. Dynamic Color Assignment (Matching your image colors)
        if 'VI' in name and 'GLCM' in name:
            bar_colors.append(colors_dict['fusion'])
        elif 'GLCM' in name:
            bar_colors.append(colors_dict['texture'])
        elif 'VI' in name:
            bar_colors.append(colors_dict['vi'])
        else:
            bar_colors.append(colors_dict['ndti'])

    # 3. Plotting
    bars = ax.bar(range(len(r2v_vals)), r2v_vals,
                  color=bar_colors, edgecolor='black', linewidth=0.8)

    # Annotate R² values above bars
    for j, v in enumerate(r2v_vals):
        ax.text(j, v + 0.01, f"{v:.2f}", ha='center', va='bottom', 
                fontsize=10, fontweight='bold')

    # Formatting
    ax.set_xticks(range(len(clean_labels)))
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylim(0, 1.1)
    
    # Title Formatting
    title_stage = stage.replace('_', ' ').title()
    if 'Ripening' in title_stage: title_stage = 'Ripening/Harvesting'
    
    ax.set_title(title_stage, fontsize=14, fontweight='bold', pad=15)
    ax.set_ylabel("Validation $R^2$" if i == 0 else "", fontsize=12)
    
    # Add a dashed line representing the Ancillary Baseline (even though bar is gone)
    # This uses the original 1st bar value as the reference line
    #ax.axhline(stage_res[0]['R2_Val'], color='black', lw=1.2, ls='--', alpha=0.4, zorder=1)
    
    ax.grid(axis='y', linestyle=':', alpha=0.3)
    for sp in ax.spines.values(): sp.set_edgecolor('black')

# Shared Legend
legend_elements = [
    Patch(facecolor=colors_dict['vi'], edgecolor='black', label='VI + Ancillary'),
    Patch(facecolor=colors_dict['texture'], edgecolor='black', label='GLCM + Ancillary'),
    Patch(facecolor=colors_dict['ndti'], edgecolor='black', label='NDTI + Ancillary'),
    Patch(facecolor=colors_dict['fusion'], edgecolor='black', label='Fusion Models')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.1), edgecolor='lightgrey')

plt.tight_layout()
save_path = output_dir / 'P_RF_Scenario_Comparison_No_First_Bar.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Graph updated: First bar (Ancillary Only) physically removed. Labels preserved.")

In [ ]:
import scipy.stats as stats

num_stages = len(winners)   # ← refresh in case winners changed

fig, axes = plt.subplots(
    num_stages, 1,
    figsize=(8, 3.5 * num_stages),
    squeeze=False
)

for i, (stage, d) in enumerate(winners.items()):

    ax  = axes[i, 0]
    lbl = stage.replace('_', '/').replace(
          'Ripening/Harvesting', 'Ripening/Harvesting')

    # Get top 10 features by RF importance
    imp_vals  = d['imp_vals']
    imp_names = d['imp_names']
    top_idx   = np.argsort(imp_vals)[::-1][:10]
    top_names = [imp_names[j] for j in top_idx]

    # Rebuild sub_df for this stage and winner scenario
    stage_df   = df[df[category_col] == stage].copy().reset_index(drop=True)
    valid_pool = [
        p for p in scenarios[d['scenario']]
        if p in stage_df.columns and stage_df[p].nunique() > 1
    ]
    sub_df = stage_df.dropna(
        subset=valid_pool + [target_var]
    ).reset_index(drop=True)

    # Compute Pearson r for each top feature vs target
    corr_vals, p_vals = [], []
    for feat in top_names:
        if feat in sub_df.columns:
            r, p = stats.pearsonr(
                sub_df[feat].astype(float),
                sub_df[target_var]
            )
            corr_vals.append(r)
            p_vals.append(p)
        else:
            corr_vals.append(0)
            p_vals.append(1)

    # Colors: positive = warm, negative = cool
    colors = []
    for r in corr_vals:
        if r >= 0:
            colors.append('#D85A30' if abs(r) > 0.5 else '#EF9F27')
        else:
            colors.append('#185FA5' if abs(r) > 0.5 else '#378ADD')

    # Plot horizontal bars
    ax.barh(
        list(reversed(top_names)),
        list(reversed(corr_vals)),
        color=list(reversed(colors)),
        edgecolor='black', linewidth=0.6
    )

    # Reference lines
    ax.axvline( 0,   color='black', lw=1.0)
    ax.axvline( 0.5, color='gray',  lw=0.7, ls='--', alpha=0.5)
    ax.axvline(-0.5, color='gray',  lw=0.7, ls='--', alpha=0.5)
    ax.axvline( 0.7, color='gray',  lw=0.7, ls=':',  alpha=0.4)
    ax.axvline(-0.7, color='gray',  lw=0.7, ls=':',  alpha=0.4)

    # Annotate r value and significance
    for j, (feat, r, p) in enumerate(
        zip(reversed(top_names), reversed(corr_vals), reversed(p_vals))
    ):
        sig  = '***' if p < 0.001 else '**' if p < 0.01 else \
               '*'   if p < 0.05  else 'ns'
        xpos = r + (0.02 if r >= 0 else -0.02)
        ha   = 'left' if r >= 0 else 'right'
        ax.text(xpos, j, f"r={r:+.2f} {sig}",
                va='center', ha=ha, fontsize=8, color='black')

    ax.set_xlim(-1.05, 1.05)
    ax.set_title(
        f"{lbl}  —  {d['scenario']}  |  top 10 features by RF importance",
        fontsize=11, fontweight='bold', color='black'
    )
    ax.set_xlabel("Pearson r with biomass", fontsize=10, color='black')
    ax.tick_params(colors='black', labelsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor('black')

# Shared legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#D85A30', edgecolor='black', label='Strong positive (|r|>0.5)'),
    Patch(facecolor='#EF9F27', edgecolor='black', label='Weak positive (|r|≤0.5)'),
    Patch(facecolor='#185FA5', edgecolor='black', label='Strong negative (|r|>0.5)'),
    Patch(facecolor='#378ADD', edgecolor='black', label='Weak negative (|r|≤0.5)'),
]
fig.legend(
    handles=legend_elements,
    loc='lower center', ncol=4,
    fontsize=9, frameon=True,
    bbox_to_anchor=(0.5, -0.01)
)

fig.text(
    0.99, 0.01,
    '*** p<0.001  ** p<0.01  * p<0.05  ns p≥0.05',
    ha='right', va='bottom', fontsize=8, color='gray'
)

plt.tight_layout(pad=2.0)
plt.subplots_adjust(bottom=0.06)

out_path = output_dir / 'Biomass_RF_MIVIF_TopFeature_Correlation.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Correlation plot saved to: {out_path}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ============================================================
# 1. SETUP STAGES & TOP 5 RF FEATURE SELECTION
# ============================================================
# Phenological order
stages_to_plot = ['Tillering', 'Booting', 'Heading', 'Ripening_Harvesting']
y_min, y_max = 0, 20   # Unified Y-axis for biomass magnitude
x_min, x_max = 0, 20
num_cols = 5            # Top 5 RF features per stage

stage_vars = {}

for stage in stages_to_plot:
    # Access winner from your winners dictionary (populated in your RF loop)
    if stage not in winners:
        continue
    
    d = winners[stage]
    imp_vals = d['imp_vals']
    imp_names = d['imp_names']
    
    # RANKING: Use the actual RF Importance values to determine the Top 5
    top_idx = np.argsort(imp_vals)[::-1][:num_cols]
    top_vars = [imp_names[j] for j in top_idx]
    
    stage_vars[stage] = top_vars

# ============================================================
# 2. GENERATE THE COMPACT UNIFIED GRID
# ============================================================
fig, axes = plt.subplots(len(stages_to_plot), num_cols, 
                         figsize=(num_cols * 3.5, len(stages_to_plot) * 3.0), 
                         squeeze=False)

# Theme: Forest Green for RF results
scatter_color = '#2d6a4f' 
line_color = '#d90429'    

for row_idx, stage in enumerate(stages_to_plot):
    current_vars = stage_vars.get(stage, [])
    row_label = stage.replace('_', ' ').replace('Harvesting', '\nHarvesting')
    
    for col_idx in range(num_cols):
        ax = axes[row_idx, col_idx]
        
        if col_idx < len(current_vars):
            var_name = current_vars[col_idx]
            # Use the full dataframe 'df' to get the raw values for the scatter
            sub_df = df[df[category_col] == stage].dropna(subset=[var_name, target_var]).copy()
            
            # 1. REGRESSION PLOT
            sns.regplot(
                data=sub_df, x=var_name, y=target_var, 
                ax=ax, color=scatter_color,
                scatter_kws={'alpha': 0.5, 's': 30, 'edgecolor': 'w', 'linewidths': 0.6},
                line_kws={'color': line_color, 'lw': 1.8, 'ls': '--'}
            )
            
            # 2. UNIFIED Y-AXIS FORMATTING (Scaling growth magnitude)
            ax.set_ylim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            
            # 3. TITLES & LABELS (X-axis is natural/auto-scaled)
            ax.set_title(f"{var_name}", fontsize=12, fontweight='bold', pad=10)
            ax.set_xlabel("Predictor Value", fontsize=12) 
            
            # Shared Y-axis labeling on far-left only
            if col_idx == 0:
                ax.set_ylabel(f"{row_label}\nBiomass (t ha$^{-1}$)", fontsize=12, fontweight='bold')
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_ylabel("")
                ax.set_yticklabels([])
                ax.tick_params(axis='y', left=False)
            
            ax.tick_params(axis='x', labelsize=10)
            
            # 4. PEARSON CORRELATION ANNOTATION
            # This shows the linear relationship of the RF-selected top features
            r_val = sub_df[var_name].corr(sub_df[target_var])
            ax.text(0.05, 0.92, f"$r={r_val:.2f}$", transform=ax.transAxes, 
                    fontsize=9, fontweight='bold', va='top', 
                    bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', boxstyle='round,pad=0.2'))
            
            ax.grid(True, linestyle=':', alpha=0.3)
        else:
            ax.axis('off')

plt.tight_layout(pad=3.0, h_pad=2.5, w_pad=1.2)

# Save the updated plot
output_path = output_dir / 'Biomass_RF_Top5_CorrelationGrid.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"RF Top 5 Feature Grid saved to {output_path}")

In [ ]:
# ============================================================
# CELL 15c — INTER-FEATURE CORRELATION HEATMAP PER STAGE
# Shows Pearson r between all top 10 features with each other
# Reveals multicollinearity patterns among RF top features
# Run after Cell 15b
# ============================================================

import math

num_stages = len(winners)
ncols = 2
nrows = math.ceil(num_stages / ncols)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(13, 11 * nrows // 2),
    squeeze=False
)

for idx in range(num_stages, nrows * ncols):
    axes[idx // ncols, idx % ncols].set_visible(False)

for i, (stage, d) in enumerate(winners.items()):

    ax  = axes[i // ncols, i % ncols]
    lbl = stage.replace('_', '/').replace(
          'Ripening/Harvesting', 'Ripening/Harvesting')

    imp_vals  = d['imp_vals']
    imp_names = d['imp_names']
    top_idx   = np.argsort(imp_vals)[::-1][:10]
    top_names = [imp_names[j] for j in top_idx]

    stage_df   = df[df[category_col] == stage].copy().reset_index(drop=True)
    valid_pool = [
        p for p in scenarios[d['scenario']]
        if p in stage_df.columns and stage_df[p].nunique() > 1
    ]
    sub_df = stage_df.dropna(
        subset=valid_pool + [target_var]
    ).reset_index(drop=True)

    top_names = [f for f in top_names if f in sub_df.columns]

    def shorten(name):
        return (name
            .replace('_GLCM_', 'GL')
            .replace('NDTI_', 'ND')
            .replace('second_moment', 'SM')
            .replace('homogeneity',   'Hom')
            .replace('dissimilarity', 'Dis')
            .replace('correlation',   'Cor')
            .replace('contrast',      'Con')
            .replace('variance',      'Var')
            .replace('entropy',       'Ent')
            .replace('mean',          'Mea')
            .replace('Green', 'G')
            .replace('NIR',   'N')
            .replace('Red',   'R')
            .replace('RE',    'RE')
        )

    short_names = [shorten(f) for f in top_names]

    # ── astype(float) handles boolean location dummy columns ──
    corr_matrix = sub_df[top_names].astype(float).corr(method='pearson')
    corr_matrix.index   = short_names
    corr_matrix.columns = short_names

    cmap = sns.diverging_palette(220, 20, as_cmap=True)

    sns.heatmap(
        corr_matrix,
        ax=ax,
        cmap=cmap,
        vmin=-1, vmax=1,
        center=0,
        annot=True,
        fmt='.2f',
        annot_kws={'size': 9},
        linewidths=0.5,
        linecolor='white',
        square=True,
        cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    )

    ax.set_title(
        f"{lbl}\n{d['scenario']}",
        fontsize=12, fontweight='bold', color='black', pad=12
    )
    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=45, ha='right', fontsize=9, color='black'
    )
    ax.set_yticklabels(
        ax.get_yticklabels(),
        rotation=0, fontsize=9, color='black'
    )
    ax.tick_params(colors='black')

plt.suptitle(
    "Inter-feature Pearson correlation — top 10 RF features per stage",
    fontsize=12, fontweight='bold', y=1.02
)

plt.tight_layout(pad=2.0)
out_path = output_dir / 'Biomass_RF_MIVIF_InterFeature_Correlation.png'
#plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Inter-feature correlation heatmap saved to: {out_path}")


# ============================================================
# BONUS — High correlation pairs (|r| > 0.80)
# ============================================================

print("\nHighly correlated feature pairs (|r| > 0.80) per stage:")
print("-" * 60)

for stage, d in winners.items():

    imp_vals  = d['imp_vals']
    imp_names = d['imp_names']
    top_idx   = np.argsort(imp_vals)[::-1][:10]
    top_names = [imp_names[j] for j in top_idx]

    stage_df   = df[df[category_col] == stage].copy().reset_index(drop=True)
    valid_pool = [
        p for p in scenarios[d['scenario']]
        if p in stage_df.columns and stage_df[p].nunique() > 1
    ]
    sub_df = stage_df.dropna(
        subset=valid_pool + [target_var]
    ).reset_index(drop=True)

    top_names = [f for f in top_names if f in sub_df.columns]

    # ── astype(float) handles boolean location dummy columns ──
    corr_mat  = sub_df[top_names].astype(float).corr(method='pearson')

    pairs_found = False
    for a in range(len(top_names)):
        for b in range(a+1, len(top_names)):
            r = corr_mat.iloc[a, b]
            if abs(r) > 0.80:
                if not pairs_found:
                    print(f"\n  {stage}:")
                    pairs_found = True
                print(f"    {top_names[a]:<40} ↔  {top_names[b]:<40}  r={r:+.3f}")

    if not pairs_found:
        print(f"\n  {stage}: no pairs with |r| > 0.80 among top 10")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.patches import Patch

# ============================================================
# CELL 16 — RPD CLASSIFICATION (UNIFIED AXIS & CLEAN LABELS)
# ============================================================

# Define chronological order
ordered_stages = ['Tillering', 'Booting', 'Heading', 'Ripening_Harvesting']
stages_found = [s for s in ordered_stages if s in set(r['Stage'] for r in results_log)]

# 1. FIND GLOBAL RPD MAX (For Unified Y-Axis)
all_rpd_vals = [r['RPD'] for r in results_log]
global_rpd_max = max(all_rpd_vals) * 1.2 if all_rpd_vals else 3.0

def rpd_color(rpd):
    if rpd < 1.4:  return '#C0392B' # Red: Unreliable
    if rpd < 1.8:  return '#E67E22' # Orange: Approximate
    if rpd < 2.0:  return '#27AE60' # Green: Acceptable
    return                '#1A5276' # Blue: Excellent

fig, axes = plt.subplots(1, len(stages_found),
                         figsize=(4.5 * len(stages_found), 6),
                         squeeze=False)

for i, stage in enumerate(stages_found):
    ax = axes[0, i]
    stage_res = [r for r in results_log if r['Stage'] == stage]
    
    # PHYSICAL REMOVAL OF 1ST BAR (Ancillary Only)
    filtered_res = stage_res[1:] if len(stage_res) > 1 else stage_res
    
    if not filtered_res:
        continue

    rpd_vals     = [r['RPD']      for r in filtered_res]
    raw_names    = [r['Scenario'] for r in filtered_res]
    bar_colors   = [rpd_color(v)  for v in rpd_vals]
    
    # AGGRESSIVE LABEL CLEANING
    clean_labels = []
    for name in raw_names:
        # Strip numbers (e.g., "2. ")
        temp = name.split('.', 1)[1].strip() if '.' in name else name
        # Remove "Ancillary" variations
        temp = temp.replace('Ancillary', '').replace('Anc', '').replace('+', '').strip()
        # Fix underscores to plus or spaces
        temp = temp.replace('_', '+').strip()
        clean_labels.append(temp)

    # Plotting
    ax.bar(range(len(rpd_vals)), rpd_vals,
           color=bar_colors, edgecolor='black', linewidth=0.7)

    # Annotate RPD values
    for j, v in enumerate(rpd_vals) :
        ax.text(j, v + 0.05, f"{v:.2f}",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Reference Thresholds
    ax.axhline(1.4, color='#E67E22', lw=1.2, ls='--', alpha=0.6)
    ax.axhline(1.8, color='#27AE60', lw=1.2, ls='--', alpha=0.6)
    ax.axhline(2.0, color='#1A5276', lw=1.2, ls='--', alpha=0.6)

    # UNIFIED AXIS FORMATTING
    ax.set_xticks(range(len(clean_labels)))
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylim(0, global_rpd_max) # Forced equal axis
    
    title_stage = stage.replace('_', ' ').title()
    if 'Ripening' in title_stage: title_stage = 'Ripening/Harvesting'
    
    ax.set_title(title_stage, fontsize=13, fontweight='bold', color='black', pad=15)
    ax.set_ylabel("RPD Score" if i == 0 else "", fontsize=12)
    ax.tick_params(labelsize=10)
    ax.grid(axis='y', linestyle=':', alpha=0.3)

# Legend
legend_elements = [
    Patch(facecolor='#C0392B', edgecolor='black', label='Unreliable (RPD < 1.4)'),
    Patch(facecolor='#E67E22', edgecolor='black', label='Approximate (1.4 ≤ RPD < 1.8)'),
    Patch(facecolor='#27AE60', edgecolor='black', label='Acceptable (1.8 ≤ RPD < 2.0)'),
    Patch(facecolor='#1A5276', edgecolor='black', label='Excellent (RPD ≥ 2.0)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.08))

plt.suptitle("",fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

#save_path = output_dir / 'Biomass_RPD_Classification_Unified.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# ============================================================
# CELL 16 — RPD CLASSIFICATION (PREMIUM AESTHETIC & FUSION LABELS)
# ============================================================

ordered_stages = ['Tillering', 'Booting', 'Heading', 'Ripening_Harvesting']
stages_found = [s for s in ordered_stages if s in set(r['Stage'] for r in results_log)]

# Set global max for unified axis
all_rpd_vals = [r['RPD'] for r in results_log]
global_rpd_max = max(all_rpd_vals) * 1.25 if all_rpd_vals else 3.0

# VIBRANT ACADEMIC PALETTE (Excellent & Bold)
def rpd_color_premium(rpd):
    if rpd < 1.4:  return '#E74C3C' # Vibrant Crimson (Unreliable)
    if rpd < 1.8:  return '#F39C12' # Rich Amber (Approximate)
    if rpd < 2.0:  return '#27AE60' # Emerald Green (Acceptable)
    return                '#2980B9' # Royal Blue (Excellent)

fig, axes = plt.subplots(1, len(stages_found),
                         figsize=(4.5 * len(stages_found), 6.5),
                         squeeze=False)

# Use serif font for that polished manuscript look
plt.rcParams['font.family'] = 'serif'

for i, stage in enumerate(stages_found):
    ax = axes[0, i]
    stage_res = [r for r in results_log if r['Stage'] == stage]
    
    # Remove 1st bar (Ancillary Only)
    filtered_res = stage_res[1:] if len(stage_res) > 1 else stage_res
    
    if not filtered_res:
        continue

    rpd_vals     = [r['RPD']      for r in filtered_res]
    raw_names    = [r['Scenario'] for r in filtered_res]
    bar_colors   = [rpd_color_premium(v) for v in rpd_vals]
    
    clean_labels = []
    for name in raw_names:
        # Strip numbers
        temp = name.split('.', 1)[1].strip() if '.' in name else name
        # Convert underscores to plus
        temp = temp.replace('_', '+')
        # FIX: Explicitly ensure it says "VI + Anc"
        temp = temp.replace('Ancillary', 'Anc').replace('Anc', 'Anc')
        if '+' not in temp and 'Anc' in temp: # Catch any single-word cases
             temp = temp.replace('Anc', ' + Anc')
        
        # Clean up spacing
        temp = " + ".join([part.strip() for part in temp.split('+')])
        clean_labels.append(temp)

    # Plotting with a slightly thicker edge for "pop"
    ax.bar(range(len(rpd_vals)), rpd_vals,
           color=bar_colors, edgecolor='black', linewidth=1.0, alpha=0.9, zorder=3)

    # Annotate RPD values - Bold and slightly larger
    for j, v in enumerate(rpd_vals):
        ax.text(j, v + 0.05, f"{v:.2f}",
                ha='center', va='bottom', fontsize=12, fontweight='bold', color='#2C3E50')

    # Threshold lines - Stylized
    ax.axhline(1.4, color='#E74C3C', lw=1.5, ls=':', alpha=0.4, zorder=2)
    ax.axhline(1.8, color='#27AE60', lw=1.5, ls=':', alpha=0.4, zorder=2)
    ax.axhline(2.0, color='#2980B9', lw=1.5, ls=':', alpha=0.4, zorder=2)

    # Formatting
    ax.set_xticks(range(len(clean_labels)))
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=12, fontweight='medium')
    ax.set_ylim(0, global_rpd_max) 
    
    title_stage = stage.replace('_', ' ').title()
    if 'Ripening' in title_stage: title_stage = 'Ripening/Harvesting'
    
    ax.set_title(title_stage, fontsize=15, fontweight='bold', color='#2C3E50', pad=20)
    ax.set_ylabel("RPD Score" if i == 0 else "", fontsize=13, fontweight='bold')
    ax.tick_params(labelsize=10)
    
    # Add a light grey background grid on Y only
    ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=0.2)
    ax.set_axisbelow(True)

# "Excellent" Legend Design
legend_elements = [
    Patch(facecolor='#E74C3C', edgecolor='black', label='Unreliable (RPD < 1.4)'),
    Patch(facecolor='#F39C12', edgecolor='black', label='Approximate (1.4–1.8)'),
    Patch(facecolor='#27AE60', edgecolor='black', label='Acceptable (1.8–2.0)'),
    Patch(facecolor='#2980B9', edgecolor='black', label='Excellent (RPD > 2.0)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=12, frameon=True, bbox_to_anchor=(0.5, -0.1), 
           edgecolor='black', shadow=True)

plt.suptitle("", fontsize=18, fontweight='bold', color='#2C3E50', y=1.05)

plt.tight_layout()
save_path = output_dir / 'Biomass_RPD.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ============================================================
# CELL 17 — STAGE × SCENARIO R²_VAL HEATMAP (STARTING @ VI + ANC)
# ============================================================

df_results = pd.DataFrame(results_log)

# 1. Pivot the data
pivot = df_results.pivot(index='Scenario', columns='Stage', values='R2_Val')

# Ensure chronological column order
ordered_stages = ['Tillering', 'Booting', 'Heading', 'Ripening_Harvesting']
stages_found = [s for s in ordered_stages if s in pivot.columns]
pivot = pivot[stages_found]

# --- PHYSICAL REMOVAL OF THE BASELINE ROW ---
# 2. Filter to remove the "Ancillary Only" row (Scenario 1)
# This forces the heatmap to start with VI + Anc
rows_to_keep = [idx for idx in pivot.index if 'Ancillary Only' not in idx and '1. ' not in idx]
pivot = pivot.loc[rows_to_keep]

# 3. CLEAN LABELS (Standardizing to VI + Anc, etc.)
clean_index = []
for idx in pivot.index:
    # Remove leading numbers (e.g., "2. ")
    temp = idx.split('.', 1)[1].strip() if '.' in idx else idx
    
    # Convert underscores to plus and standardize Ancillary to "Anc"
    temp = temp.replace('_', '+').replace('Ancillary', 'Anc')
    
    # Ensure the " + Anc" spacing is perfect
    if 'Anc' in temp and '+' not in temp:
        temp = temp.replace('Anc', ' + Anc')
    
    # Final cleanup of double spaces/pluses
    temp = temp.replace('++', '+').replace('  ', ' ').strip()
    clean_index.append(temp)

pivot.index = clean_index

# 4. PLOTTING
fig, ax = plt.subplots(figsize=(10, 6))

# Using your preferred "RdYlGn" Colormap
cmap = sns.color_palette("RdYlGn", as_cmap=True)

sns.heatmap(
    pivot, ax=ax, cmap=cmap,
    vmin=0.0, vmax=1.0,
    annot=True, fmt='.2f',
    annot_kws={'size': 12, 'weight': 'bold', 'family': 'serif'},
    linewidths=1.5, linecolor='white',
    cbar_kws={'label': 'Validation $R^2$', 'shrink': 0.9}
)

# 5. HIGHLIGHT THE BEST SCENARIO PER STAGE
for col_idx, stage in enumerate(pivot.columns):
    best_row = pivot[stage].idxmax()
    row_idx  = list(pivot.index).index(best_row)
    # Add a bold black rectangle around the winning cell
    ax.add_patch(plt.Rectangle(
        (col_idx, row_idx), 1, 1,
        fill=False, edgecolor='black', lw=3.5, zorder=5
    ))

# 6. FORMATTING
ax.set_title(
    "",
    fontsize=14, fontweight='bold', family='serif', pad=25
)

# Format X-axis (Stages)
ax.set_xticklabels(
    [s.replace('_', '/').replace('Ripening/Harvesting', 'Ripening/\nHarvesting').title()
     for s in pivot.columns],
    rotation=0, fontsize=12, fontweight='bold', family='serif'
)

# Format Y-axis (Scenarios - Starting from VI + Anc)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, 
                   fontsize=12, fontweight='bold', family='serif')

ax.set_xlabel("Wheat Growth Stage", fontsize=12, fontweight='bold', family='serif')
ax.set_ylabel("Data Fusion Scenarios", fontsize=12, fontweight='bold', family='serif')

plt.tight_layout()

# Save the final heatmap
save_path = output_dir / 'Biomass_Heatmap.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Heatmap finalized: Starting from VI + Anc. Saved to: {save_path}")

In [ ]:
# ============================================================
# CELL 18 — FEATURE SELECTION STABILITY PLOT
# ============================================================
 
fig, axes = plt.subplots(len(winners), 1,
                         figsize=(9, 3.8 * len(winners)),
                         squeeze=False)
 
for i, (stage, d) in enumerate(winners.items()):
    ax  = axes[i, 0]
    lbl = stage.replace('_', '/')
 
    stage_result = next(
        (r for r in results_log
         if r['Stage'] == stage and r['Scenario'] == d['scenario']),
        None
    )
 
    if stage_result is None or 'Stable_Features' not in stage_result:
        ax.set_visible(False)
        continue
 
    stable_str   = stage_result['Stable_Features']
    stable_feats = [f.strip() for f in stable_str.split(',') if f.strip()]
 
    imp_vals  = d['imp_vals']
    imp_names = d['imp_names']
 
    stable_imp = []
    for feat in stable_feats:
        if feat in imp_names:
            idx = imp_names.index(feat)
            stable_imp.append(imp_vals[idx])
        else:
            stable_imp.append(0.0)
 
    sorted_pairs = sorted(zip(stable_feats, stable_imp),
                          key=lambda x: x[1])
    sorted_feats = [p[0] for p in sorted_pairs]
    sorted_imp   = [p[1] for p in sorted_pairs]
 
    colors = []
    for feat in sorted_feats:
        if feat.startswith('loc_') or feat in ['DAP', 'GDD']:
            colors.append('#888780')
        elif 'NDTI' in feat:
            colors.append('#534AB7')
        elif 'GLCM' in feat:
            colors.append('#378ADD')
        else:
            colors.append('#D85A30')
 
    ax.barh(sorted_feats, sorted_imp,
            color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(
        f"{lbl}  —  {d['scenario']}  |  "
        f"Stable features (selected ≥3 of 5 folds)",
        fontsize=11, fontweight='bold', color='black'
    )
    ax.set_xlabel("Mean decrease in impurity", fontsize=10, color='black')
    ax.tick_params(colors='black', labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor('black')
 
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#D85A30', edgecolor='black', label='Vegetation Index'),
    Patch(facecolor='#378ADD', edgecolor='black', label='GLCM Texture'),
    Patch(facecolor='#534AB7', edgecolor='black', label='NDTI Texture'),
    Patch(facecolor='#888780', edgecolor='black', label='Ancillary (DAP/GDD)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.01))
 
plt.suptitle(
    "",
    fontsize=12, fontweight='bold', y=0.8
)
plt.tight_layout(pad=2.0)
plt.subplots_adjust(bottom=0.06)
out_path = output_dir / 'Biomass_RF_MIVIF_FeatureStability.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Cell 18 saved: {out_path}")

In [ ]:
import rasterio
from rasterio.enums import Resampling
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor

# ============================================================
# BIOMASS SPATIAL MAPPING — RF
# Uses best scenario stable features per stage
# Refits RF on full stage data, predicts pixel-wise
# ============================================================

map_output_folder = Path(r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Python1\Result_Chapter_3\RF\Biomass_maps')
map_output_folder.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. CONFIGURATION
# ============================================================
base_path = r"C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Drone1\Kentland"

# Target variable — Biomass
target_var_map = 'Biomass_density(t_ha)'      # FIX: change to your exact biomass column name in df

stages_to_run = {
    'Tillering':            'Tillering',
    'Booting':              'Booting',
    'Heading':              'Heading',
    'Ripening_Harvesting':  'Harvesting',
}

# FIX: Match these to your main RF loop hyperparameters
rf_params = {
    'n_estimators':      500,
    'max_depth':         10,
    'min_samples_split': 10,
    'min_samples_leaf':  10,
    'max_features':      'sqrt',
    'random_state':      42,
    'n_jobs':            -1
}

NODATA_VAL = -9999.0    # standard nodata — 0 is a valid biomass value

# ============================================================
# 2. MAPPING LOOP
# ============================================================
for stage, folder_name in stages_to_run.items():
    print(f"\n{'='*55}")
    print(f"  PROCESSING: {stage}")
    print(f"{'='*55}")

    # A. Get winning scenario stable features from results_log
    stage_results = [r for r in results_log if r['Stage'] == stage]
    if not stage_results:
        print(f"  SKIP: No results found for {stage} in results_log")
        continue

    winner_log      = max(stage_results, key=lambda x: x['R2_Val'])
    stable_features = [f.strip() for f in winner_log['Stable_Features'].split(',') if f.strip()]

    if not stable_features:
        print(f"  SKIP: No stable features for {stage}")
        continue

    print(f"  Scenario : {winner_log['Scenario']}")
    print(f"  R²_val   : {winner_log['R2_Val']:.3f}")
    print(f"  Features : {stable_features}")

    # B. Refit RF on full stage data using stable features
    stage_df = df[df[category_col] == stage].dropna(
        subset=stable_features + [target_var_map]
    ).copy()

    if len(stage_df) < 10:
        print(f"  SKIP: Only {len(stage_df)} training samples — too few for {stage}")
        continue

    rf_map_model = RandomForestRegressor(**rf_params)
    rf_map_model.fit(stage_df[stable_features], stage_df[target_var_map])
    print(f"  Model refitted on {len(stage_df)} samples.")

    input_folder    = os.path.join(base_path, folder_name)
    output_map_path = map_output_folder / f"Kentland_{folder_name}_Biomass_Map.tif"

    try:
        # C. Master template from first stable feature
        master_feat = os.path.join(input_folder, f"{stable_features[0]}.tif")
        if not os.path.exists(master_feat):
            master_feat = master_feat.replace(".tif", ".TIF")
        if not os.path.exists(master_feat):
            print(f"  ERROR: Master raster not found: {master_feat}")
            continue

        with rasterio.open(master_feat) as src:
            meta         = src.meta.copy()
            master_shape = (src.height, src.width)
            print(f"  Raster shape : {master_shape}")
            print(f"  CRS          : {src.crs}")

        # D. Stack all feature bands with auto-resize to master shape
        data_stack    = []
        missing_feats = []

        for feat in stable_features:
            f_path = os.path.join(input_folder, f"{feat}.tif")
            if not os.path.exists(f_path):
                f_path = f_path.replace(".tif", ".TIF")
            if not os.path.exists(f_path):
                print(f"  WARNING: Raster not found for '{feat}' — filling with zeros")
                data_stack.append(
                    np.zeros(master_shape[0] * master_shape[1], dtype=np.float32)
                )
                missing_feats.append(feat)
                continue

            with rasterio.open(f_path) as s:
                if (s.height, s.width) != master_shape:
                    print(f"  Resampling '{feat}' {(s.height, s.width)} → {master_shape}")
                    band_data = s.read(
                        1,
                        out_shape=master_shape,
                        resampling=Resampling.bilinear
                    )
                else:
                    band_data = s.read(1)

                # Replace nodata with NaN before stacking
                band_data = band_data.astype(np.float32)
                if s.nodata is not None:
                    band_data[band_data == s.nodata] = np.nan

                data_stack.append(band_data.flatten())

        if missing_feats:
            print(f"  WARNING: {len(missing_feats)} missing feature(s): {missing_feats}")

        # E. Predict pixel-wise
        X_spatial = np.column_stack(data_stack)   # (n_pixels, n_features)

        # Mask: only finite pixels (NaN = nodata or edge)
        mask = np.all(np.isfinite(X_spatial), axis=1)

        predictions = np.full(X_spatial.shape[0], NODATA_VAL, dtype=np.float32)

        if np.any(mask):
            print(f"  Predicting {np.sum(mask):,} valid pixels "
                  f"({np.sum(~mask):,} masked as nodata)...")
            raw_preds = rf_map_model.predict(X_spatial[mask]).astype(np.float32)

            # Clip to 0 — biomass cannot be negative
            predictions[mask] = np.clip(raw_preds, 0, None)
        else:
            print(f"  WARNING: No valid pixels found for {stage}")

        # F. Reshape and save GeoTIFF
        biomass_map = predictions.reshape(master_shape)

        meta.update(
            count=1,
            dtype='float32',
            nodata=NODATA_VAL,
        )

        with rasterio.open(str(output_map_path), 'w', **meta) as dst:
            dst.write(biomass_map, 1)

        # Summary stats on valid pixels
        valid_preds = predictions[predictions != NODATA_VAL]
        if len(valid_preds) > 0:
            print(f"  Map stats — Min: {valid_preds.min():.4f}  "
                  f"Max: {valid_preds.max():.4f}  "
                  f"Mean: {valid_preds.mean():.4f}  kg m⁻²")

        print(f"  SUCCESS: Map saved → {output_map_path}")

    except Exception as e:
        import traceback
        print(f"  ERROR for {stage}: {e}")
        traceback.print_exc()

print(f"\n{'='*55}")
print(f"  Output folder: {map_output_folder.resolve()}")
print(f"{'='*55}")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import geopandas as gpd

# ============================================================
# VISUALIZE BIOMASS MAPS — Greens colormap + field boundary
# ============================================================

plt.rcParams['font.family'] = 'serif'

shapefile_path = r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Boundary\WGS_84_UTM_17\Boundary\Kentland_georef1.shp'

stage_map_order = [
    ('Tillering',           'Tillering'),
    ('Booting',             'Booting'),
    ('Heading',             'Heading'),
    ('Ripening_Harvesting', 'Harvesting'),
]

# ==========================================
# 1. COLLECT MAP PATHS
# ==========================================
map_paths = []
for stage, folder_name in stage_map_order:
    p = map_output_folder / f"Kentland_{folder_name}_Biomass_Map.tif"
    if p.exists():
        map_paths.append((stage, folder_name, p))
    else:
        print(f"WARNING: Map not found for {stage} — {p}")

n_maps = len(map_paths)

if n_maps == 0:
    print("No maps found. Run the mapping loop first.")
else:
    # ==========================================
    # 2. LOAD & REPROJECT BOUNDARY ONCE
    # ==========================================
    boundary_gdf = gpd.read_file(shapefile_path)
    print(f"Boundary CRS : {boundary_gdf.crs}")

    # ==========================================
    # 3. READ RASTERS
    # ==========================================
    NODATA   = -9999.0
    cmap     = plt.get_cmap('Greens')   # ← Greens colormap
    vmin, vmax = 0, 20                  # ← fixed 0–20 ton/ha scale

    raster_data = {}
    for stage, folder_name, path in map_paths:
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr == NODATA] = np.nan
            raster_data[stage] = {
                'arr':       arr,
                'transform': src.transform,
                'crs':       src.crs,
                'path':      path,
            }

    print(f"Color scale: {vmin} — {vmax} ton ha⁻¹  (fixed scale)")

    # ==========================================
    # 4. PLOT
    # ==========================================
    fig, axes = plt.subplots(
        1, n_maps,
        figsize=(5.0 * n_maps, 6),
        squeeze=False,
        gridspec_kw={'wspace': 0.05}
    )

    for i, (stage, folder_name, path) in enumerate(map_paths):
        ax  = axes[0, i]
        d   = raster_data[stage]
        arr = d['arr']
        lbl = stage.replace('_', ' ').title()

        # Show raster
        im = ax.imshow(
            arr,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            interpolation='nearest'
        )

        # ----------------------------------------
        # BOUNDARY OVERLAY
        # Reproject boundary to raster CRS if needed
        # then convert geo coords → pixel coords
        # ----------------------------------------
        raster_crs = d['crs']
        if boundary_gdf.crs != raster_crs:
            boundary_plot = boundary_gdf.to_crs(raster_crs)
        else:
            boundary_plot = boundary_gdf

        with rasterio.open(path) as src:
            transform = src.transform

        for _, brow in boundary_plot.iterrows():
            geom = brow.geometry
            if geom is None:
                continue

            # Handle both Polygon and MultiPolygon
            from shapely.geometry import MultiPolygon
            polys = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]

            for poly in polys:
                # Exterior ring
                coords = list(poly.exterior.coords)
                cols_px = [(x - transform.c) / transform.a for x, y in coords]
                rows_px = [(y - transform.f) / transform.e for x, y in coords]
                ax.plot(cols_px, rows_px,
                        color='black', lw=1.5, zorder=5)

                # Interior rings (holes) if any
                for interior in poly.interiors:
                    coords = list(interior.coords)
                    cols_px = [(x - transform.c) / transform.a for x, y in coords]
                    rows_px = [(y - transform.f) / transform.e for x, y in coords]
                    ax.plot(cols_px, rows_px,
                            color='black', lw=1.0, ls='--', zorder=5)

        ax.set_title(lbl, fontsize=13, fontweight='bold', pad=8)
        ax.axis('off')

        # Colorbar below each panel
        divider = make_axes_locatable(ax)
        cax     = divider.append_axes("bottom", size="5%", pad=0.08)
        cb      = plt.colorbar(im, cax=cax, orientation='horizontal')
        cb.set_ticks([0, 5, 10, 15, 20])
        cb.set_label("Biomass (ton ha$^{-1}$)", fontsize=11)
        cb.ax.tick_params(labelsize=10)

    plt.suptitle(
        "",
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout(pad=1.5, w_pad=0.5)

    viz_path = map_output_folder / 'Kentland_Biomass_Maps_Overview1.png'
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Map saved to: {viz_path.resolve()}")

In [ ]:
import rasterio
from rasterio.enums import Resampling
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor

# ============================================================
# BIOMASS SPATIAL MAPPING — RF
# Uses best scenario stable features per stage
# Refits RF on full stage data, predicts pixel-wise
# ============================================================

map_output_folder = Path(r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Python1\Result_Chapter_3\RF\Biomass_maps')
map_output_folder.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. CONFIGURATION
# ============================================================
base_path = r"C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Drone1\Orange"

# Target variable — Biomass
target_var_map = 'Biomass_density(t_ha)'      # FIX: change to your exact biomass column name in df

stages_to_run = {
    'Tillering':            'Tillering',
    'Booting':              'Booting',
    'Heading':              'Heading',
    'Ripening_Harvesting':  'Harvesting',
}

# FIX: Match these to your main RF loop hyperparameters
rf_params = {
    'n_estimators':      500,
    'max_depth':         10,
    'min_samples_split': 10,
    'min_samples_leaf':  10,
    'max_features':      'sqrt',
    'random_state':      42,
    'n_jobs':            -1
}

NODATA_VAL = -9999.0    # standard nodata — 0 is a valid biomass value

# ============================================================
# 2. MAPPING LOOP
# ============================================================
for stage, folder_name in stages_to_run.items():
    print(f"\n{'='*55}")
    print(f"  PROCESSING: {stage}")
    print(f"{'='*55}")

    # A. Get winning scenario stable features from results_log
    stage_results = [r for r in results_log if r['Stage'] == stage]
    if not stage_results:
        print(f"  SKIP: No results found for {stage} in results_log")
        continue

    winner_log      = max(stage_results, key=lambda x: x['R2_Val'])
    stable_features = [f.strip() for f in winner_log['Stable_Features'].split(',') if f.strip()]

    if not stable_features:
        print(f"  SKIP: No stable features for {stage}")
        continue

    print(f"  Scenario : {winner_log['Scenario']}")
    print(f"  R²_val   : {winner_log['R2_Val']:.3f}")
    print(f"  Features : {stable_features}")

    # B. Refit RF on full stage data using stable features
    stage_df = df[df[category_col] == stage].dropna(
        subset=stable_features + [target_var_map]
    ).copy()

    if len(stage_df) < 10:
        print(f"  SKIP: Only {len(stage_df)} training samples — too few for {stage}")
        continue

    rf_map_model = RandomForestRegressor(**rf_params)
    rf_map_model.fit(stage_df[stable_features], stage_df[target_var_map])
    print(f"  Model refitted on {len(stage_df)} samples.")

    input_folder    = os.path.join(base_path, folder_name)
    output_map_path = map_output_folder / f"Orange_{folder_name}_Biomass_Map.tif"

    try:
        # C. Master template from first stable feature
        master_feat = os.path.join(input_folder, f"{stable_features[0]}.tif")
        if not os.path.exists(master_feat):
            master_feat = master_feat.replace(".tif", ".TIF")
        if not os.path.exists(master_feat):
            print(f"  ERROR: Master raster not found: {master_feat}")
            continue

        with rasterio.open(master_feat) as src:
            meta         = src.meta.copy()
            master_shape = (src.height, src.width)
            print(f"  Raster shape : {master_shape}")
            print(f"  CRS          : {src.crs}")

        # D. Stack all feature bands with auto-resize to master shape
        data_stack    = []
        missing_feats = []

        for feat in stable_features:
            f_path = os.path.join(input_folder, f"{feat}.tif")
            if not os.path.exists(f_path):
                f_path = f_path.replace(".tif", ".TIF")
            if not os.path.exists(f_path):
                print(f"  WARNING: Raster not found for '{feat}' — filling with zeros")
                data_stack.append(
                    np.zeros(master_shape[0] * master_shape[1], dtype=np.float32)
                )
                missing_feats.append(feat)
                continue

            with rasterio.open(f_path) as s:
                if (s.height, s.width) != master_shape:
                    print(f"  Resampling '{feat}' {(s.height, s.width)} → {master_shape}")
                    band_data = s.read(
                        1,
                        out_shape=master_shape,
                        resampling=Resampling.bilinear
                    )
                else:
                    band_data = s.read(1)

                # Replace nodata with NaN before stacking
                band_data = band_data.astype(np.float32)
                if s.nodata is not None:
                    band_data[band_data == s.nodata] = np.nan

                data_stack.append(band_data.flatten())

        if missing_feats:
            print(f"  WARNING: {len(missing_feats)} missing feature(s): {missing_feats}")

        # E. Predict pixel-wise
        X_spatial = np.column_stack(data_stack)   # (n_pixels, n_features)

        # Mask: only finite pixels (NaN = nodata or edge)
        mask = np.all(np.isfinite(X_spatial), axis=1)

        predictions = np.full(X_spatial.shape[0], NODATA_VAL, dtype=np.float32)

        if np.any(mask):
            print(f"  Predicting {np.sum(mask):,} valid pixels "
                  f"({np.sum(~mask):,} masked as nodata)...")
            raw_preds = rf_map_model.predict(X_spatial[mask]).astype(np.float32)

            # Clip to 0 — biomass cannot be negative
            predictions[mask] = np.clip(raw_preds, 0, None)
        else:
            print(f"  WARNING: No valid pixels found for {stage}")

        # F. Reshape and save GeoTIFF
        biomass_map = predictions.reshape(master_shape)

        meta.update(
            count=1,
            dtype='float32',
            nodata=NODATA_VAL,
        )

        with rasterio.open(str(output_map_path), 'w', **meta) as dst:
            dst.write(biomass_map, 1)

        # Summary stats on valid pixels
        valid_preds = predictions[predictions != NODATA_VAL]
        if len(valid_preds) > 0:
            print(f"  Map stats — Min: {valid_preds.min():.4f}  "
                  f"Max: {valid_preds.max():.4f}  "
                  f"Mean: {valid_preds.mean():.4f}  kg m⁻²")

        print(f"  SUCCESS: Map saved → {output_map_path}")

    except Exception as e:
        import traceback
        print(f"  ERROR for {stage}: {e}")
        traceback.print_exc()

print(f"\n{'='*55}")
print(f"  Output folder: {map_output_folder.resolve()}")
print(f"{'='*55}")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import geopandas as gpd

# ============================================================
# VISUALIZE BIOMASS MAPS — Greens colormap + field boundary
# ============================================================

plt.rcParams['font.family'] = 'serif'

shapefile_path = r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Boundary\WGS_84_UTM_17\Boundary\Orange_georef.shp'

stage_map_order = [
    ('Tillering',           'Tillering'),
    ('Booting',             'Booting'),
    ('Heading',             'Heading'),
    ('Ripening_Harvesting', 'Harvesting'),
]

# ==========================================
# 1. COLLECT MAP PATHS
# ==========================================
map_paths = []
for stage, folder_name in stage_map_order:
    p = map_output_folder / f"Orange_{folder_name}_Biomass_Map.tif"
    if p.exists():
        map_paths.append((stage, folder_name, p))
    else:
        print(f"WARNING: Map not found for {stage} — {p}")

n_maps = len(map_paths)

if n_maps == 0:
    print("No maps found. Run the mapping loop first.")
else:
    # ==========================================
    # 2. LOAD & REPROJECT BOUNDARY ONCE
    # ==========================================
    boundary_gdf = gpd.read_file(shapefile_path)
    print(f"Boundary CRS : {boundary_gdf.crs}")

    # ==========================================
    # 3. READ RASTERS
    # ==========================================
    NODATA   = -9999.0
    cmap     = plt.get_cmap('Greens')   # ← Greens colormap
    vmin, vmax = 0, 20                  # ← fixed 0–20 ton/ha scale

    raster_data = {}
    for stage, folder_name, path in map_paths:
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr == NODATA] = np.nan
            raster_data[stage] = {
                'arr':       arr,
                'transform': src.transform,
                'crs':       src.crs,
                'path':      path,
            }

    print(f"Color scale: {vmin} — {vmax} ton ha⁻¹  (fixed scale)")

    # ==========================================
    # 4. PLOT
    # ==========================================
    fig, axes = plt.subplots(
        1, n_maps,
        figsize=(5.0 * n_maps, 6),
        squeeze=False,
    )
    #gridspec_kw={'wspace': 0.01}

    for i, (stage, folder_name, path) in enumerate(map_paths):
        ax  = axes[0, i]
        d   = raster_data[stage]
        arr = d['arr']
        lbl = stage.replace('_', ' ').title()

        # Show raster
        im = ax.imshow(
            arr,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            interpolation='nearest'
        )

        # ----------------------------------------
        # BOUNDARY OVERLAY
        # Reproject boundary to raster CRS if needed
        # then convert geo coords → pixel coords
        # ----------------------------------------
        raster_crs = d['crs']
        if boundary_gdf.crs != raster_crs:
            boundary_plot = boundary_gdf.to_crs(raster_crs)
        else:
            boundary_plot = boundary_gdf

        with rasterio.open(path) as src:
            transform = src.transform

        for _, brow in boundary_plot.iterrows():
            geom = brow.geometry
            if geom is None:
                continue

            # Handle both Polygon and MultiPolygon
            from shapely.geometry import MultiPolygon
            polys = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]

            for poly in polys:
                # Exterior ring
                coords = list(poly.exterior.coords)
                cols_px = [(x - transform.c) / transform.a for x, y in coords]
                rows_px = [(y - transform.f) / transform.e for x, y in coords]
                ax.plot(cols_px, rows_px,
                        color='black', lw=1.5, zorder=5)

                # Interior rings (holes) if any
                for interior in poly.interiors:
                    coords = list(interior.coords)
                    cols_px = [(x - transform.c) / transform.a for x, y in coords]
                    rows_px = [(y - transform.f) / transform.e for x, y in coords]
                    ax.plot(cols_px, rows_px,
                            color='black', lw=1.0, ls='--', zorder=5)

        ax.set_title(lbl, fontsize=13, fontweight='bold', pad=8)
        ax.axis('off')

        # Colorbar below each panel
        divider = make_axes_locatable(ax)
        cax     = divider.append_axes("bottom", size="4%", pad=0.08)
        cb      = plt.colorbar(im, cax=cax, orientation='horizontal')
        cb.set_ticks([0, 5, 10, 15, 20])
        cb.set_label("Biomass (ton ha$^{-1}$)", fontsize=11)
        cb.ax.tick_params(labelsize=10)

    plt.suptitle(
        "",
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout(pad=1.5, w_pad=0.2)

    viz_path = map_output_folder / 'Orange_Biomass_Maps_Overview.png'
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Map saved to: {viz_path.resolve()}")

In [ ]:
import rasterio
from rasterio.enums import Resampling
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor

# ============================================================
# BIOMASS SPATIAL MAPPING — RF
# Uses best scenario stable features per stage
# Refits RF on full stage data, predicts pixel-wise
# ============================================================

map_output_folder = Path(r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Python1\Result_Chapter_3\RF\Biomass_maps')
map_output_folder.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. CONFIGURATION
# ============================================================
base_path = r"C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Drone1\Augusta"

# Target variable — Biomass
target_var_map = 'Biomass_density(t_ha)'      # FIX: change to your exact biomass column name in df

stages_to_run = {
    'Tillering':            'Tillering',
    'Booting':              'Booting',
    'Heading':              'Heading',
    'Ripening_Harvesting':  'Harvesting',
}

# FIX: Match these to your main RF loop hyperparameters
rf_params = {
    'n_estimators':      500,
    'max_depth':         10,
    'min_samples_split': 10,
    'min_samples_leaf':  10,
    'max_features':      'sqrt',
    'random_state':      42,
    'n_jobs':            -1
}

NODATA_VAL = -9999.0    # standard nodata — 0 is a valid biomass value

# ============================================================
# 2. MAPPING LOOP
# ============================================================
for stage, folder_name in stages_to_run.items():
    print(f"\n{'='*55}")
    print(f"  PROCESSING: {stage}")
    print(f"{'='*55}")

    # A. Get winning scenario stable features from results_log
    stage_results = [r for r in results_log if r['Stage'] == stage]
    if not stage_results:
        print(f"  SKIP: No results found for {stage} in results_log")
        continue

    winner_log      = max(stage_results, key=lambda x: x['R2_Val'])
    stable_features = [f.strip() for f in winner_log['Stable_Features'].split(',') if f.strip()]

    if not stable_features:
        print(f"  SKIP: No stable features for {stage}")
        continue

    print(f"  Scenario : {winner_log['Scenario']}")
    print(f"  R²_val   : {winner_log['R2_Val']:.3f}")
    print(f"  Features : {stable_features}")

    # B. Refit RF on full stage data using stable features
    stage_df = df[df[category_col] == stage].dropna(
        subset=stable_features + [target_var_map]
    ).copy()

    if len(stage_df) < 10:
        print(f"  SKIP: Only {len(stage_df)} training samples — too few for {stage}")
        continue

    rf_map_model = RandomForestRegressor(**rf_params)
    rf_map_model.fit(stage_df[stable_features], stage_df[target_var_map])
    print(f"  Model refitted on {len(stage_df)} samples.")

    input_folder    = os.path.join(base_path, folder_name)
    output_map_path = map_output_folder / f"Augusta_{folder_name}_Biomass_Map.tif"

    try:
        # C. Master template from first stable feature
        master_feat = os.path.join(input_folder, f"{stable_features[0]}.tif")
        if not os.path.exists(master_feat):
            master_feat = master_feat.replace(".tif", ".TIF")
        if not os.path.exists(master_feat):
            print(f"  ERROR: Master raster not found: {master_feat}")
            continue

        with rasterio.open(master_feat) as src:
            meta         = src.meta.copy()
            master_shape = (src.height, src.width)
            print(f"  Raster shape : {master_shape}")
            print(f"  CRS          : {src.crs}")

        # D. Stack all feature bands with auto-resize to master shape
        data_stack    = []
        missing_feats = []

        for feat in stable_features:
            f_path = os.path.join(input_folder, f"{feat}.tif")
            if not os.path.exists(f_path):
                f_path = f_path.replace(".tif", ".TIF")
            if not os.path.exists(f_path):
                print(f"  WARNING: Raster not found for '{feat}' — filling with zeros")
                data_stack.append(
                    np.zeros(master_shape[0] * master_shape[1], dtype=np.float32)
                )
                missing_feats.append(feat)
                continue

            with rasterio.open(f_path) as s:
                if (s.height, s.width) != master_shape:
                    print(f"  Resampling '{feat}' {(s.height, s.width)} → {master_shape}")
                    band_data = s.read(
                        1,
                        out_shape=master_shape,
                        resampling=Resampling.bilinear
                    )
                else:
                    band_data = s.read(1)

                # Replace nodata with NaN before stacking
                band_data = band_data.astype(np.float32)
                if s.nodata is not None:
                    band_data[band_data == s.nodata] = np.nan

                data_stack.append(band_data.flatten())

        if missing_feats:
            print(f"  WARNING: {len(missing_feats)} missing feature(s): {missing_feats}")

        # E. Predict pixel-wise
        X_spatial = np.column_stack(data_stack)   # (n_pixels, n_features)

        # Mask: only finite pixels (NaN = nodata or edge)
        mask = np.all(np.isfinite(X_spatial), axis=1)

        predictions = np.full(X_spatial.shape[0], NODATA_VAL, dtype=np.float32)

        if np.any(mask):
            print(f"  Predicting {np.sum(mask):,} valid pixels "
                  f"({np.sum(~mask):,} masked as nodata)...")
            raw_preds = rf_map_model.predict(X_spatial[mask]).astype(np.float32)

            # Clip to 0 — biomass cannot be negative
            predictions[mask] = np.clip(raw_preds, 0, None)
        else:
            print(f"  WARNING: No valid pixels found for {stage}")

        # F. Reshape and save GeoTIFF
        biomass_map = predictions.reshape(master_shape)

        meta.update(
            count=1,
            dtype='float32',
            nodata=NODATA_VAL,
        )

        with rasterio.open(str(output_map_path), 'w', **meta) as dst:
            dst.write(biomass_map, 1)

        # Summary stats on valid pixels
        valid_preds = predictions[predictions != NODATA_VAL]
        if len(valid_preds) > 0:
            print(f"  Map stats — Min: {valid_preds.min():.4f}  "
                  f"Max: {valid_preds.max():.4f}  "
                  f"Mean: {valid_preds.mean():.4f}  kg m⁻²")

        print(f"  SUCCESS: Map saved → {output_map_path}")

    except Exception as e:
        import traceback
        print(f"  ERROR for {stage}: {e}")
        traceback.print_exc()

print(f"\n{'='*55}")
print(f"  Output folder: {map_output_folder.resolve()}")
print(f"{'='*55}")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import geopandas as gpd

# ============================================================
# VISUALIZE BIOMASS MAPS — Greens colormap + field boundary
# ============================================================

plt.rcParams['font.family'] = 'serif'

shapefile_path = r'C:\Users\sheetalk22\OneDrive\HP Drive D\Virginia_Tech\Fall_2025\Research\Boundary\WGS_84_UTM_17\Boundary\Augusta_Re.shp'

stage_map_order = [
    ('Tillering',           'Tillering'),
    ('Booting',             'Booting'),
    ('Heading',             'Heading'),
    ('Ripening_Harvesting', 'Harvesting'),
]

# ==========================================
# 1. COLLECT MAP PATHS
# ==========================================
map_paths = []
for stage, folder_name in stage_map_order:
    p = map_output_folder / f"Augusta_{folder_name}_Biomass_Map.tif"
    if p.exists():
        map_paths.append((stage, folder_name, p))
    else:
        print(f"WARNING: Map not found for {stage} — {p}")

n_maps = len(map_paths)

if n_maps == 0:
    print("No maps found. Run the mapping loop first.")
else:
    # ==========================================
    # 2. LOAD & REPROJECT BOUNDARY ONCE
    # ==========================================
    boundary_gdf = gpd.read_file(shapefile_path)
    print(f"Boundary CRS : {boundary_gdf.crs}")

    # ==========================================
    # 3. READ RASTERS
    # ==========================================
    NODATA   = -9999.0
    cmap     = plt.get_cmap('Greens')   # ← Greens colormap
    vmin, vmax = 0, 20                  # ← fixed 0–20 ton/ha scale

    raster_data = {}
    for stage, folder_name, path in map_paths:
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr == NODATA] = np.nan
            raster_data[stage] = {
                'arr':       arr,
                'transform': src.transform,
                'crs':       src.crs,
                'path':      path,
            }

    print(f"Color scale: {vmin} — {vmax} ton ha⁻¹  (fixed scale)")

    # ==========================================
    # 4. PLOT
    # ==========================================
    fig, axes = plt.subplots(
        1, n_maps,
        figsize=(5.0 * n_maps, 6),
        squeeze=False,
    )
    #gridspec_kw={'wspace': 0.01}

    for i, (stage, folder_name, path) in enumerate(map_paths):
        ax  = axes[0, i]
        d   = raster_data[stage]
        arr = d['arr']
        lbl = stage.replace('_', ' ').title()

        # Show raster
        im = ax.imshow(
            arr,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            interpolation='nearest'
        )

        # ----------------------------------------
        # BOUNDARY OVERLAY
        # Reproject boundary to raster CRS if needed
        # then convert geo coords → pixel coords
        # ----------------------------------------
        raster_crs = d['crs']
        if boundary_gdf.crs != raster_crs:
            boundary_plot = boundary_gdf.to_crs(raster_crs)
        else:
            boundary_plot = boundary_gdf

        with rasterio.open(path) as src:
            transform = src.transform

        for _, brow in boundary_plot.iterrows():
            geom = brow.geometry
            if geom is None:
                continue

            # Handle both Polygon and MultiPolygon
            from shapely.geometry import MultiPolygon
            polys = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]

            for poly in polys:
                # Exterior ring
                coords = list(poly.exterior.coords)
                cols_px = [(x - transform.c) / transform.a for x, y in coords]
                rows_px = [(y - transform.f) / transform.e for x, y in coords]
                ax.plot(cols_px, rows_px,
                        color='black', lw=1.5, zorder=5)

                # Interior rings (holes) if any
                for interior in poly.interiors:
                    coords = list(interior.coords)
                    cols_px = [(x - transform.c) / transform.a for x, y in coords]
                    rows_px = [(y - transform.f) / transform.e for x, y in coords]
                    ax.plot(cols_px, rows_px,
                            color='black', lw=1.0, ls='--', zorder=5)

        ax.set_title(lbl, fontsize=13, fontweight='bold', pad=8)
        ax.axis('off')

        # Colorbar below each panel
        divider = make_axes_locatable(ax)
        cax     = divider.append_axes("bottom", size="4%", pad=0.08)
        cb      = plt.colorbar(im, cax=cax, orientation='horizontal')
        cb.set_ticks([0, 5, 10, 15, 20])
        cb.set_label("Biomass (ton ha$^{-1}$)", fontsize=11)
        cb.ax.tick_params(labelsize=10)

    plt.suptitle(
        "",
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout(pad=1.5, w_pad=0.2)

    viz_path = map_output_folder / 'Augusta_Biomass_Maps_Overview.png'
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Map saved to: {viz_path.resolve()}")